In [ ]:
# cleaning all the 3 datasets

In [1]:
import os
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

ROOT = "/home/jovyan/project" # root direcotry
IN = ROOT + "/data/parquet"   # input directory
OUT = ROOT + "/data/clean"    # output directory

spark = (SparkSession.builder.appName("dpa-clean").config("spark.driver.memory", "2g").config("spark.sql.session.timeZone", "UTC").getOrCreate())

# saving the count for every step
counts = []


def save_count(table, step, df):
    n = df.count()
    counts.append((table, step, n))
    print(step, ":", n)

In [2]:
# Flights data

In [3]:
flights = spark.read.parquet(IN + "/flights")

# filtering out the columns
column_list = ["Year", "Month", "FlightDate", "DayOfWeek", "Reporting_Airline", "Tail_Number",
"Flight_Number_Reporting_Airline", "Origin", "Dest", "CRSDepTime", "CRSArrTime",
"DepDelay", "ArrDelay", "ArrDel15", "Cancelled", "Diverted", "AirTime", "Distance",
"CarrierDelay", "WeatherDelay", "NASDelay", "LateAircraftDelay"]
flights = flights.select(column_list)
save_count("flights", "raw data", flights)
flights.show(5)

raw data : 13926960
+----+-----+----------+---------+-----------------+-----------+-------------------------------+------+----+----------+----------+--------+--------+--------+---------+--------+-------+--------+------------+------------+--------+-----------------+
|Year|Month|FlightDate|DayOfWeek|Reporting_Airline|Tail_Number|Flight_Number_Reporting_Airline|Origin|Dest|CRSDepTime|CRSArrTime|DepDelay|ArrDelay|ArrDel15|Cancelled|Diverted|AirTime|Distance|CarrierDelay|WeatherDelay|NASDelay|LateAircraftDelay|
+----+-----+----------+---------+-----------------+-----------+-------------------------------+------+----+----------+----------+--------+--------+--------+---------+--------+-------+--------+------------+------------+--------+-----------------+
|2024|    5|2024-05-30|        4|               NK|     N676NK|                           1175|   CLT| ORD|      0953|      1108|     0.0|     5.0|     0.0|      0.0|     0.0|   96.0|   599.0|        NULL|        NULL|    NULL|             NU

In [4]:
flights.printSchema()

root
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- FlightDate: string (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- Tail_Number: string (nullable = true)
 |-- Flight_Number_Reporting_Airline: integer (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- CRSDepTime: string (nullable = true)
 |-- CRSArrTime: string (nullable = true)
 |-- DepDelay: double (nullable = true)
 |-- ArrDelay: double (nullable = true)
 |-- ArrDel15: double (nullable = true)
 |-- Cancelled: double (nullable = true)
 |-- Diverted: double (nullable = true)
 |-- AirTime: double (nullable = true)
 |-- Distance: double (nullable = true)
 |-- CarrierDelay: double (nullable = true)
 |-- WeatherDelay: double (nullable = true)
 |-- NASDelay: double (nullable = true)
 |-- LateAircraftDelay: double (nullable = true)



In [5]:
# missing values
flights.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in flights.columns]).show(vertical=True)

-RECORD 0-----------------------------------
 Year                            | 0        
 Month                           | 0        
 FlightDate                      | 0        
 DayOfWeek                       | 0        
 Reporting_Airline               | 0        
 Tail_Number                     | 35638    
 Flight_Number_Reporting_Airline | 1        
 Origin                          | 0        
 Dest                            | 0        
 CRSDepTime                      | 0        
 CRSArrTime                      | 0        
 DepDelay                        | 177499   
 ArrDelay                        | 218310   
 ArrDel15                        | 218310   
 Cancelled                       | 0        
 Diverted                        | 0        
 AirTime                         | 218310   
 Distance                        | 0        
 CarrierDelay                    | 11090298 
 WeatherDelay                    | 11090298 
 NASDelay                        | 11090298 
 LateAircr

In [6]:
flights.select("DepDelay", "ArrDelay", "AirTime", "Distance").describe().show()

+-------+------------------+-----------------+------------------+-----------------+
|summary|          DepDelay|         ArrDelay|           AirTime|         Distance|
+-------+------------------+-----------------+------------------+-----------------+
|  count|          13749461|         13708650|          13708650|         13926960|
|   mean|12.484421825699204|6.866497357507851|114.90103766599921|833.5695934360406|
| stddev|55.701758079581936|57.54479032950202| 70.54244416639062|597.7922498832229|
|    min|             -99.0|           -126.0|               5.0|             11.0|
|    max|            4413.0|           4405.0|             724.0|           5095.0|
+-------+------------------+-----------------+------------------+-----------------+



In [7]:
flights.groupBy("Cancelled", "Diverted").count().show()

+---------+--------+--------+
|Cancelled|Diverted|   count|
+---------+--------+--------+
|      0.0|     1.0|   34051|
|      1.0|     0.0|  184258|
|      0.0|     0.0|13708651|
+---------+--------+--------+



In [8]:
# duplicate flights based on same date, airline, flight number, origin, destination and departure time

In [9]:
key = ["FlightDate", "Reporting_Airline", "Flight_Number_Reporting_Airline", "Origin", "Dest", "CRSDepTime"]
flights = flights.dropDuplicates(key)
save_count("flights", "remove duplicates", flights)

remove duplicates : 13926960


In [10]:
flights = flights.filter((F.col("Cancelled") == 0) & (F.col("Diverted") == 0))
save_count("flights", "remove cancelled and diverted", flights)

remove cancelled and diverted : 13708651


In [11]:
# remove flights with missing delays or missing tail number

In [12]:
flights = flights.withColumn("Tail_Number", F.upper(F.trim(F.col("Tail_Number"))))
flights = flights.filter(F.col("DepDelay").isNotNull() & F.col("ArrDelay").isNotNull())
flights = flights.filter(F.col("Tail_Number").isNotNull() & (F.col("Tail_Number") != ""))


In [13]:
save_count("flights", "remove missing delay or tail number", flights)

remove missing delay or tail number : 13708650


In [14]:
# remove impossible values like the distance or air time of 0

In [15]:
flights = flights.filter((F.col("Distance") > 0) & (F.col("AirTime") > 0))
save_count("flights", "remove impossible values", flights)

remove impossible values : 13708650


In [16]:
# create the departure hour based on CRSDepTime
# 0545 hour is the first 2 characters

In [17]:
flights = flights.withColumn("dep_hour", F.substring(F.lpad("CRSDepTime", 4, "0"), 1, 2).cast("int"))
print("flights with hour 24:", flights.filter(F.col("dep_hour") == 24).count())
flights = flights.withColumn("dep_hour", F.when(F.col("dep_hour") == 24, 0).otherwise(F.col("dep_hour")))
flights.select("FlightDate", "CRSDepTime", "dep_hour", "Origin", "Dest", "ArrDelay").show(5)

flights with hour 24: 1
+----------+----------+--------+------+----+--------+
|FlightDate|CRSDepTime|dep_hour|Origin|Dest|ArrDelay|
+----------+----------+--------+------+----+--------+
|2024-08-25|      0600|       6|   MIA| ATL|    -5.0|
|2023-09-16|      0530|       5|   DEN| DAL|   -10.0|
|2024-11-23|      0540|       5|   JFK| FLL|   -23.0|
|2024-11-27|      0545|       5|   JFK| FLL|   -23.0|
|2023-09-10|      0549|       5|   JFK| LAX|    -8.0|
+----------+----------+--------+------+----+--------+
only showing top 5 rows



In [18]:
flights.write.mode("overwrite").partitionBy("Year", "Month").parquet(OUT + "/flights")

In [51]:
# Airports data cleaning

In [19]:
airports = spark.read.parquet(IN + "/airports")
save_count("airports", "raw data", airports)
airports.show(5)

raw data : 86242
+------+-----+-------------+--------------------+------------------+------------------+------------+---------+-----------+----------+------------+-----------------+---------+---------+--------+----------+---------+--------------------+--------------------+
|    id|ident|         type|                name|      latitude_deg|     longitude_deg|elevation_ft|continent|iso_country|iso_region|municipality|scheduled_service|icao_code|iata_code|gps_code|local_code|home_link|      wikipedia_link|            keywords|
+------+-----+-------------+--------------------+------------------+------------------+------------+---------+-----------+----------+------------+-----------------+---------+---------+--------+----------+---------+--------------------+--------------------+
|325143| FVLS|small_airport|   Lonestar Airfield|        -21.061103|         31.889759|        1303|       AF|         ZW|     ZW-MV|    Chiredzi|               no|     NULL|     NULL|    FVLS|      FVLS|     NUL

In [22]:
airports.printSchema()


root
 |-- id: integer (nullable = true)
 |-- ident: string (nullable = true)
 |-- type: string (nullable = true)
 |-- name: string (nullable = true)
 |-- latitude_deg: double (nullable = true)
 |-- longitude_deg: double (nullable = true)
 |-- elevation_ft: integer (nullable = true)
 |-- continent: string (nullable = true)
 |-- iso_country: string (nullable = true)
 |-- iso_region: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- scheduled_service: string (nullable = true)
 |-- icao_code: string (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- gps_code: string (nullable = true)
 |-- local_code: string (nullable = true)
 |-- home_link: string (nullable = true)
 |-- wikipedia_link: string (nullable = true)
 |-- keywords: string (nullable = true)



In [23]:
airports.groupBy("type").count().orderBy(F.desc("count")).show()

+--------------+-----+
|          type|count|
+--------------+-----+
| small_airport|42812|
|      heliport|23240|
|        closed|13574|
|medium_airport| 4104|
| seaplane_base| 1273|
| large_airport| 1176|
|   balloonport|   63|
+--------------+-----+



In [24]:
us = airports.filter(F.col("iso_country") == "US")
print("US airports:", us.count())
print("US airports with an IATA code:", us.filter(F.col("iata_code").isNotNull()).count())

US airports: 32694
US airports with an IATA code: 2036


In [25]:
# keeping only US airports that are not closed and have an IATA code( this code is used in flights data)
airports = airports.filter((F.col("iso_country") == "US") & (F.col("type") != "closed"))
airports = airports.filter(F.col("iata_code").isNotNull())
save_count("airports", "keep open US airports with IATA code", airports)


keep open US airports with IATA code : 2036


In [26]:
airports = airports.dropDuplicates(["iata_code"])
save_count("airports", "remove duplicate IATA codes", airports)

remove duplicate IATA codes : 2036


In [27]:
column_list = ["iata_code", "icao_code", "name", "type", "municipality", "iso_region", "latitude_deg", "longitude_deg", "elevation_ft"]
airports = airports.select(column_list)
airports.write.mode("overwrite").parquet(OUT + "/airports")
airports.show(5)

+---------+---------+--------------------+--------------+-------------------+----------+-------------+------------------+------------+
|iata_code|icao_code|                name|          type|       municipality|iso_region| latitude_deg|     longitude_deg|elevation_ft|
+---------+---------+--------------------+--------------+-------------------+----------+-------------+------------------+------------+
|      AAF|     KAAF|Apalachicola Regi...| small_airport|       Apalachicola|     US-FL|    29.727501|        -85.027496|          20|
|      ABE|     KABE|Lehigh Valley Int...|medium_airport|Allentown/Bethlehem|     US-PA|    40.651773|        -75.442797|         393|
|      ABI|     KABI|Abilene Regional ...|medium_airport|            Abilene|     US-TX|32.4113006592|-99.68190002440001|        1791|
|      ABL|     PAFM|      Ambler Airport|medium_airport|             Ambler|     US-AK|    67.105529|       -157.855339|         334|
|      ABQ|     KABQ|Albuquerque Inter...| large_airpor

In [54]:
# Weather

In [28]:
weather = spark.read.parquet(IN + "/weather")
save_count("weather", "raw data", weather)
weather.select("STATION", "DATE", "REPORT_TYPE", "WND", "CIG", "VIS", "TMP", "AA1").show(5, truncate=False)

raw data : 1263091
+-----------+-------------------+-----------+--------------+-----------+------------+-------+-----------+
|STATION    |DATE               |REPORT_TYPE|WND           |CIG        |VIS         |TMP    |AA1        |
+-----------+-------------------+-----------+--------------+-----------+------------+-------+-----------+
|72537094847|2024-01-01T00:00:00|FM-12      |240,1,N,0031,1|99999,9,9,N|012000,1,9,9|+0033,1|06,0000,9,1|
|72537094847|2024-01-01T00:13:00|FM-16      |260,5,N,0031,5|00396,5,M,N|012875,5,N,5|+0033,5|01,0000,2,1|
|72537094847|2024-01-01T00:53:00|FM-15      |240,5,N,0026,5|00457,5,M,N|011265,5,N,5|+0028,5|01,0000,2,5|
|72537094847|2024-01-01T01:53:00|FM-15      |290,5,N,0021,5|01067,5,M,N|012875,5,N,5|+0028,5|01,0000,9,5|
|72537094847|2024-01-01T02:13:00|FM-16      |330,5,N,0026,5|00396,5,M,N|012875,5,N,5|+0028,5|NULL       |
+-----------+-------------------+-----------+--------------+-----------+------------+-------+-----------+
only showing top 5 rows



In [29]:
# kinds of reports are in the files
weather.groupBy("REPORT_TYPE").count().orderBy(F.desc("count")).show()

+-----------+------+
|REPORT_TYPE| count|
+-----------+------+
|      FM-15|858829|
|      FM-12|200183|
|      FM-16|164019|
|      SOD  | 35406|
|      SHEF |  2598|
|      SOM  |  1158|
|      SY-MT|   890|
|      MESOW|     8|
+-----------+------+



In [30]:
weather.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in weather.columns]).show(vertical=True)

-RECORD 0-------------
 STATION     | 0      
 DATE        | 0      
 REPORT_TYPE | 0      
 CALL_SIGN   | 0      
 WND         | 0      
 CIG         | 0      
 VIS         | 0      
 TMP         | 0      
 DEW         | 0      
 AA1         | 290428 



In [31]:
weather.groupBy("STATION").count().orderBy("count").show(60)

+-----------+-----+
|    STATION|count|
+-----------+-----+
|72494523293|19871|
|72483993225|19907|
|74783012849|20765|
|72493023230|21014|
|72258013960|21312|
|72244012918|22360|
|72297793184|22407|
|72210812894|22582|
|72278023183|23382|
|72386023169|23556|
|91182022521|23930|
|72534014819|24551|
|72494023234|25149|
|72502014734|25231|
|72306013722|25291|
|72205012815|25410|
|72243012960|25442|
|72406093721|25763|
|72403093738|25796|
|72219013874|25970|
|72202012839|26362|
|72428014821|26388|
|72254013904|26440|
|74486094789|26471|
|72698024229|26473|
|72434013994|26645|
|72572024127|26675|
|72211012842|26710|
|72259003927|26859|
|72421093814|26862|
|72295023174|26877|
|72327013897|26878|
|72520094823|26930|
|72503014732|27061|
|72231012916|27260|
|72314013881|27297|
|72530094846|27330|
|72446003947|27343|
|72408013739|27410|
|72658014922|27430|
|72793024233|27492|
|72253012921|27498|
|72290023188|27506|
|72565003017|27573|
|72438093819|28181|
|72509014739|28435|
|72405013743|28801|


In [32]:
# keeping only the hourly reports FM-15

In [33]:
weather = weather.filter(F.trim(F.col("REPORT_TYPE")) == "FM-15")
save_count("weather", "keep hourly reports FM-15", weather)

keep hourly reports FM-15 : 858829


In [34]:
# splitting the text fileds into numbers 
weather = (weather.withColumn("wind_speed", F.split("WND", ",")[3].cast("double") / 10).withColumn("ceiling", F.split("CIG", ",")[0].cast("double"))
           .withColumn("visibility", F.split("VIS", ",")[0].cast("double"))
           .withColumn("temp", F.split("TMP", ",")[0].cast("double") / 10)
           .withColumn("rain", F.split("AA1", ",")[1].cast("double") / 10))

In [35]:
# replacing the missing codes like 9999

In [36]:
weather = (weather.withColumn("wind_speed", F.when(F.col("wind_speed") != 999.9, F.col("wind_speed")))
           .withColumn("ceiling", F.when(F.col("ceiling") != 99999, F.col("ceiling")))
           .withColumn("visibility", F.when(F.col("visibility") != 999999, F.col("visibility")))
           .withColumn("temp", F.when(F.col("temp") != 999.9, F.col("temp")))
           .withColumn("rain", F.when(F.col("rain") != 999.9, F.col("rain"))))

for c in ["wind_speed", "ceiling", "visibility", "temp", "rain"]:
    print(c, "missing:", weather.filter(F.col(c).isNull()).count())

wind_speed missing: 1027
ceiling missing: 778
visibility missing: 396
temp missing: 1198
rain missing: 27611


In [37]:
# rounding the time to hours 

In [38]:
weather = weather.withColumn("time", F.to_timestamp("DATE", "yyyy-MM-dd'T'HH:mm:ss"))
weather = weather.withColumn("hour_utc", F.date_trunc("hour", F.col("time") + F.expr("INTERVAL 30 MINUTES")))
weather = weather.dropDuplicates(["STATION", "hour_utc"])
save_count("weather", "one report per station and hour", weather)

one report per station and hour : 857812


In [39]:
# marking the bad weather

In [40]:
weather = weather.withColumn("bad_weather",(F.col("visibility") < 4828) | (F.col("ceiling") < 305)| (F.col("wind_speed") >= 10.3) | (F.col("rain") >= 2.5))
weather = weather.fillna({"bad_weather": False})


In [41]:
weather = weather.select("STATION", "hour_utc", "wind_speed", "ceiling", "visibility","temp", "rain", "bad_weather")
weather.write.mode("overwrite").parquet(OUT + "/weather")
weather.groupBy("bad_weather").count().show()

+-----------+------+
|bad_weather| count|
+-----------+------+
|       true| 62737|
|      false|795075|
+-----------+------+



In [42]:
# summary of cleaning

In [43]:
summary = pd.DataFrame(counts, columns=["table", "step", "rows_left"])
# rows removed = rows left after the step before - rows left now
summary["rows_removed"] = -summary.groupby("table")["rows_left"].diff().fillna(0).astype(int)
print(summary)
os.makedirs(ROOT + "/results", exist_ok=True)
summary.to_csv(ROOT + "/results/cleaning_counts.csv", index=False)

       table                                  step  rows_left  rows_removed
0    flights                              raw data   13926960             0
1    flights                     remove duplicates   13926960             0
2    flights         remove cancelled and diverted   13708651        218309
3    flights   remove missing delay or tail number   13708650             1
4    flights              remove impossible values   13708650             0
5   airports                              raw data      86242             0
6   airports  keep open US airports with IATA code       2036         84206
7   airports           remove duplicate IATA codes       2036             0
8    weather                              raw data    1263091             0
9    weather             keep hourly reports FM-15     858829        404262
10   weather       one report per station and hour     857812          1017
